# Ôn tập Buổi 05 - Missing Data và Combine

        **Thời lượng gợi ý:** 60-75 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Chọn chiến lược missing dựa trên ý nghĩa dữ liệu.
- Phân biệt `concat` với `merge/join`.
- Kiểm tra tính toàn vẹn khóa khi ghép bảng.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi5_python_datascience.pdf`.


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import numpy as np
import pandas as pd
STATE_DIR = ROOT / 'datasets' / 'buoi5' / 'state'
BABY_DIR = ROOT / 'datasets' / 'buoi5' / 'babynames_sample'
print(f'Repo: {ROOT}')


## 0. Chẩn đoán nhanh - chưa chạy code

**1. Có `NaN` thì luôn `dropna()` đúng không?**

<details><summary>Kiểm tra đáp án</summary>

Không. Phải xét nguyên nhân thiếu, vai trò cột và chi phí mất dòng.

</details>

**2. `concat` và `merge` khác nhau ở đâu?**

<details><summary>Kiểm tra đáp án</summary>

`concat` ghép theo trục; `merge` khớp bản ghi theo key.

</details>

**3. Vì sao `validate='many_to_one'` hữu ích?**

<details><summary>Kiểm tra đáp án</summary>

Nó phát hiện key ở bảng phía `one` bị trùng, tránh nhân số dòng ngoài ý muốn.

</details>


## 1. Audit missing trước khi xử lý


In [ ]:
students = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "score": [8.0, np.nan, 7.5, np.nan, 9.0],
    "city": ["HCM", "HN", None, "HCM", "HN"],
})
missing_report = students.isna().agg(["sum", "mean"]).T
missing_report.columns = ["missing_count", "missing_rate"]
display(missing_report)
assert missing_report.loc["score", "missing_count"] == 2


## 2. Drop hay fill phải có lý do

Ví dụ minh họa dùng median cho `score` vì ít nhạy với cực trị hơn mean. Đây không phải quy tắc áp cho mọi dataset.


In [ ]:
cleaned = students.copy()
cleaned["score"] = cleaned["score"].fillna(cleaned["score"].median())
cleaned["city"] = cleaned["city"].fillna("Unknown")
display(cleaned)
assert cleaned.isna().sum().sum() == 0


## 3. MultiIndex, `stack()` và `unstack()`


In [ ]:
series = pd.Series(
    [100, 110, 90, 95],
    index=pd.MultiIndex.from_product([["HCM", "HN"], [2025, 2026]], names=["city", "year"]),
    name="population_index",
)
wide = series.unstack("year")
long_again = wide.stack().rename("population_index")
display(wide)
assert long_again.sort_index().equals(series.sort_index())


## 4. `concat`: gộp nhiều file cùng schema


In [ ]:
pieces = []
for path in sorted(BABY_DIR.glob("yob*.txt")):
    year = int(path.stem.removeprefix("yob"))
    piece = pd.read_csv(path, names=["name", "gender", "births"])
    piece["year"] = year
    pieces.append(piece)
baby_names = pd.concat(pieces, ignore_index=True)
print(baby_names.shape, sorted(baby_names["year"].unique()))
assert baby_names["year"].nunique() == len(pieces) == 4


## 5. `merge`: case study US State Population


In [ ]:
population = pd.read_csv(STATE_DIR / "state-population.csv")
areas = pd.read_csv(STATE_DIR / "state-areas.csv")
abbrevs = pd.read_csv(STATE_DIR / "state-abbrevs.csv")

merged = population.merge(
    abbrevs,
    how="left",
    left_on="state/region",
    right_on="abbreviation",
    validate="many_to_one",
    indicator=True,
)
print(merged["_merge"].value_counts())
display(merged.loc[merged["state"].isna(), ["state/region"]].drop_duplicates())
assert len(merged) == len(population)


Hai mã `PR` và `USA` không có trong bảng abbreviation; cần quyết định tường minh thay vì để missing âm thầm. Sau khi ghép diện tích, dòng tổng hợp toàn nước (`USA`) không phải một bang và không có diện tích trong bảng `state-areas`, nên phải được báo và loại khỏi phép tính mật độ.


In [ ]:
merged["state"] = merged["state"].fillna(
    merged["state/region"].map({"PR": "Puerto Rico", "USA": "United States"})
)
final = merged.drop(columns=["abbreviation", "_merge"]).merge(
    areas, how="left", on="state", validate="many_to_one"
)
candidates_2010 = final.query("year == 2010 and ages == 'total'")
excluded = candidates_2010.loc[
    candidates_2010[["population", "area (sq. mi)"]].isna().any(axis=1),
    ["state", "population", "area (sq. mi)"],
]
print("Không đủ dữ liệu để tính density:")
display(excluded)
density_2010 = candidates_2010.dropna(subset=["population", "area (sq. mi)"]).assign(
    density=lambda d: d["population"] / d["area (sq. mi)"]
)
display(density_2010.nlargest(5, "density")[["state", "density"]])
assert density_2010["density"].notna().all()


## Bài tự luyện

        Từ `baby_names`, tính tổng số trẻ theo `year` và `gender`, rồi chuyển thành bảng rộng có `gender` ở cột.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        answer = (
    baby_names.groupby(["year", "gender"], as_index=False)["births"].sum()
    .pivot(index="year", columns="gender", values="births")
)
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi không xử lý missing trước khi hiểu ngữ nghĩa.
- [ ] Tôi chọn được `concat` hay `merge` theo bài toán.
- [ ] Tôi dùng `validate`, `indicator` và kiểm tra số dòng sau merge.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
